In [1]:
import pandas as pd 
import numpy as np 
import scanpy as sc
import matplotlib.pyplot as plt
import concurrent.futures
import pickle
import warnings
from datetime import date
import hisepy
import os
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed,ProcessPoolExecutor
from tqdm import tqdm
import anndata
import gc
warnings.filterwarnings("ignore")
sc.settings.n_jobs = 60
print("Current working directory:", os.getcwd())

Current working directory: /home/workspace/pediatric_IH/jupyter/Analysis/DEG/Flu_Y2_D0_D7


# Read Files and filter genes for DESEQ2

In [2]:
#read in total adata 
adata = sc.read_h5ad('/home/workspace/pediatric_IH/jupyter/certpro_up1/Cleaned_sample_analysis/up1_deepclean_CMV_COVID.h5ad')

In [4]:
adata

AnnData object with n_obs × n_vars = 3541330 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden_2', 'doublets_manual', 'cmv_status', 'relabeled_v

In [3]:
subset_adata = adata[adata.obs['sample.visitName'].isin(['Flu Year 2 Pre-Vac 7-12 Weeks','Flu Year 2 Day 7']), :].copy()

In [6]:
len(set(subset_adata.obs['subject.subjectGuid']))

20

In [8]:
set(subset_adata.obs['sample.visitName'])

{'Flu Year 2 Day 90', 'Flu Year 2 Pre-Vac 7-12 Weeks'}

In [9]:
fitlered_gene=pd.DataFrame()
for i in subset_adata.obs['AIFI_L3'].unique():
    print(i)
    adata_subset=subset_adata[subset_adata.obs['AIFI_L3']==i]
    sc.pp.filter_genes(adata_subset, min_cells=round(np.shape(adata_subset.X)[0]*0.1))
    gene_list=pd.DataFrame(list(adata_subset.var.index))
    gene_list.columns=['gene']
    gene_list['AIFI_L3']=i
    fitlered_gene=pd.concat([fitlered_gene,gene_list])

CLP cell
Adaptive NK cell
GZMK+ CD56dim NK cell
Proliferating T cell
Core CD16 monocyte
SOX4+ naive CD8 T cell
CD4 MAIT
ISG+ naive B cell
CD95 memory B cell
ISG+ memory CD8 T cell
Core naive CD4 T cell
cDC1
CD8 MAIT
CM CD8 T cell
CD8aa
Core memory B cell
KLRF1- GZMB+ CD27- EM CD8 T cell
CM CD4 T cell
Plasma cell
CD56bright NK cell
Core naive CD8 T cell
SOX4+ naive CD4 T cell
ISG+ MAIT
ILC
CD14+ cDC2
Early memory B cell
HLA-DRhi cDC2
GZMK+ memory CD4 Treg
GZMK- CD27+ EM CD8 T cell
KLRF1+ effector Vd1 gdT
GZMK+ Vd2 gdT
KLRB1+ memory CD8 Treg
pDC
GZMB- CD27- EM CD4 T cell
ISG+ CD56dim NK cell
SOX4+ Vd1 gdT
Naive Vd1 gdT
Type 2 polarized memory B cell
KLRF1+ GZMB+ CD27- EM CD8 T cell
ISG+ cDC2
Core naive B cell
Core CD14 monocyte
Transitional B cell
Proliferating NK cell
ISG+ naive CD4 T cell
Erythrocyte
Platelet
ASDC
CD27- effector B cell
ISG+ CD16 monocyte
GZMK+ CD27+ EM CD8 T cell
Intermediate monocyte
Activated memory B cell
CMP cell
CD27+ effector B cell
ISG+ naive CD8 T cell
KLRF1- G

In [13]:
fitlered_gene

,gene,AIFI_L3
0,NOC2L,CLP cell
1,ISG15,CLP cell
2,TNFRSF18,CLP cell
3,TNFRSF4,CLP cell
4,SDF4,CLP cell
...,...,...
4295,MT-ND4L,GZMB- CD27+ EM CD4 T cell
4296,MT-ND4,GZMB- CD27+ EM CD4 T cell
4297,MT-ND5,GZMB- CD27+ EM CD4 T cell
4298,MT-ND6,GZMB- CD27+ EM CD4 T cell


In [10]:
fitlered_gene.to_csv("UP1_filtered_gene_Y2_D0_D90_clean.csv")

In [26]:
fitlered_gene.head()

,gene,AIFI_L3
0,NOC2L,CD8 MAIT
1,ISG15,CD8 MAIT
2,SDF4,CD8 MAIT
3,UBE2J2,CD8 MAIT
4,INTS11,CD8 MAIT
